# Utiliser des LLM en API pour générer avec une prompt (zero shot)

- des annotations : quels sont les articles qui mentionnent des thématiques IA
- de l'extraction d'information : quelles sont les notions utilisées

Quelques lectures 

- Stuhler, O., Ton, C. D., & Ollion, E. (2025). From Codebooks to Promptbooks: Extracting Information from Text with Generative Large Language Models. Sociological Methods & Research, 54(3), 794-848. https://doi.org/10.1177/00491241251336794 (Original work published 2025)
- https://www.css.cnrs.fr/classification-with-generative-llms-and-api-calls/

## Quelques mots sur le génératif

- Prédiction du next token
- Modèles pré-entrainés
    - des tailles & alignements différents
- Dépendances matérielles fortes

## Quels modèles ?

- vaste question
- équilibre puissance/coût (financier et matériel)
- enjeu de la sécurité des données

De nombreux prestataires

- openai, etc.
- openrouter
- humanum...

## Charger les données

In [3]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/pyshs/URFIST-Lyon-2026/refs/heads/main/data/css_openalex_26022026.csv")
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")
df["texte"] = df["title"]+ df["abstract"]
df = df[(df["texte"].str.len() > 100) & (df["texte"].str.len() < 5000)]
df.head(3)

,id,type,primary_location,title,abstract_inverted_index,publication_year,publication_date,open_access,relevance_score,abstract,journal,texte
0,https://openalex.org/W2159397589,article,"{'id': 'doi:10.1126/science.1167742', 'is_oa':...",Computational Social Science,"{'A': [0], 'field': [1], 'is': [2], 'emerging'...",2009.0,2009-02-06,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",1360.35770,A field is emerging that leverages the capacit...,Science,Computational Social ScienceA field is emergin...
2,https://openalex.org/W3081158114,article,"{'id': 'doi:10.1126/science.aaz8170', 'is_oa':...",Computational social science: Obstacles and op...,"{'Data': [0], 'sharing,': [1], 'research': [2]...",2020.0,2020-08-28,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",438.53986,"Data sharing, research ethics, and incentives ...",Science,Computational social science: Obstacles and op...
3,https://openalex.org/W3022499311,article,{'id': 'doi:10.1146/annurev-soc-121919-054621'...,Computational Social Science and Sociology,"{'The': [0], 'integration': [1], 'of': [2, 16,...",2020.0,2020-04-28,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",413.03424,The integration of social science with compute...,Annual Review of Sociology,Computational Social Science and SociologyThe ...


## Charger une clé

In [5]:
import yaml
with open("../creds.yaml", "r") as f:
    config = yaml.safe_load(f)
api_key = config["openrouter"]

## Utiliser openrouter et l'API openapi

Quickstart : https://platform.openai.com/docs/quickstart?api-mode=chat&language=python

In [9]:
#!pip install openai
from openai import OpenAI
api_url = "https://openrouter.ai/api/v1"
 
client = OpenAI(
  base_url=api_url,
  api_key=api_key,
)

response = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct",
    messages=[{"role": "user", "content": "Quel est ton avis sur les URFIST en France ?"}]
)
response

ChatCompletion(id='gen-1774624079-X2J4JEKt1URATiML72QE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Les URFIST (Unités Régionales de Formation à l'Information Scientifique et Technique) en France jouent un rôle crucial dans la promotion de la culture scientifique, technique et de l'information auprès des publics variés, notamment dans le contexte de l'éducation et de la recherche. Voici quelques aspects qui reflètent leur importance et leur impact :\n\n1. **Formation et éducation** : Les URFIST proposent des formations et des ateliers pour enseignants, étudiants, et chercheurs sur des sujets tels que la recherche d'information, l'évaluation des sources, la méthodologie de la recherche, et l'utilisation des ressources documentaires. Ces formations sont essentielles pour desarroller les compétences nécessaires dans un monde où l'information est abondante et nécessite une approche critique.\n\n2. **Appui à la recherche** : Les URFI

Choisir un modèle

In [14]:
type(response)

openai.types.chat.chat_completion.ChatCompletion

In [16]:
print(response.choices[0].message.content)

Les URFIST (Unités Régionales de Formation à l'Information Scientifique et Technique) en France jouent un rôle crucial dans la promotion de la culture scientifique, technique et de l'information auprès des publics variés, notamment dans le contexte de l'éducation et de la recherche. Voici quelques aspects qui reflètent leur importance et leur impact :

1. **Formation et éducation** : Les URFIST proposent des formations et des ateliers pour enseignants, étudiants, et chercheurs sur des sujets tels que la recherche d'information, l'évaluation des sources, la méthodologie de la recherche, et l'utilisation des ressources documentaires. Ces formations sont essentielles pour desarroller les compétences nécessaires dans un monde où l'information est abondante et nécessite une approche critique.

2. **Appui à la recherche** : Les URFIST offrent un soutien aux chercheurs et aux étudiants en leur fournissant des outils et des méthodes pour mener des recherches efficaces. Cela inclut l'accès à de

Construire la prompt

In [21]:
messages = [
  {
    "role": "system",
    "content": "You are an efficient research assistant helping with text annotation."
  },
  {
    "role": "user",
    "content": """
    Annotate this following text if its topic is about AI or algorithms.
    If so, returns AI, and otherwise, return NOT AI.
    Do not return anything else than AI or NOT AI
    """
      + df["texte"].dropna().iloc[1]
  }
]

In [22]:
#!pip install openai
from openai import OpenAI
api_url = "https://openrouter.ai/api/v1"
 
client = OpenAI(
  base_url=api_url,
  api_key=api_key,
)

response = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct",
    messages=messages
)
response.choices[0].message.content

'NOT AI'

### Application Faire une boucle sur une liste de documents : évaluer si un abstract porte ou pas sur l'IA

- Tester avec des exemples (few shot)
- Tester avec du raisonnement (chaîne of thought)

In [25]:
api_url = "https://openrouter.ai/api/v1"
 
client = OpenAI(
  base_url=api_url,
  api_key=api_key,
)

results = []
c = 0

for texte in df["texte"][0:10]:
    print(c)
    
    messages = [
  {
    "role": "system",
    "content": "You are an efficient research assistant helping with text annotation."
  },
  {
    "role": "user",
    "content": """
    Annotate this following text if its topic is about AI or algorithms.
    If so, returns AI, and otherwise, return NOT AI.
    Do not return anything else than AI or NOT AI
    """
          + texte
      }
    ]

    response = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct",
    messages=messages
    )
    response = response.choices[0].message.content
    results.append(response)
    c += 1

In [26]:
results

['AI', 'NOT AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI']

## Evaluation

Comment évaluer la qualité d'un codage ? 

- La nécessité d'un gold standard
- Le calcul de métriques `from sklearn.metrics import classification_report`

In [39]:
truth = ['NOT AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI']
results = ['NOT AI', 'AI', 'NOT AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI']

### Application : construire un jeu de données d'évaluation

- Utiliser des outils permettant l'interaction humaine
- Toujours de l'ambiguité : accord inter-annotateur (alpha de Chronbach, autre)

### Mesurer la qualité d'une évaluation

Différentes métriques

- précision
- recall
- f1

In [30]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

In [29]:
accuracy_score(truth, results)

0.8

In [38]:
truth

['NOT AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI', 'AI']

In [42]:
print(classification_report(truth, results))

              precision    recall  f1-score   support

          AI       1.00      0.89      0.94         9
      NOT AI       0.50      1.00      0.67         1

    accuracy                           0.90        10
   macro avg       0.75      0.94      0.80        10
weighted avg       0.95      0.90      0.91        10



## Application : extraire de l'information

Sur les articles identifiés comme AI-related, extraire les termes liés pour les analyser

- Construire une prompt
- Tester sur plusieurs textes
- Gérer la structure des données